# EDA - yellow trip data

Notebook analizuje zbiór danych NYC Yellow Taxi


In [2]:
import pandas as pd
from pathlib import Path

## Settings

In [45]:
FILE_PATH = Path("data/raw_download/yellow_tripdata_2024-01.parquet")

pd.set_option("display.max_columns", None)      # show all columns
pd.set_option("display.width", 200)             # show more characters in a row
pd.set_option("display.max_rows", 100)          # show more rows

print(FILE_PATH)
print("File exists:", FILE_PATH.exists())

file_size_bytes = FILE_PATH.stat().st_size
file_size_mb = file_size_bytes / 1024**2

print(f"Size: {file_size_mb:.2f} MB")

data/raw_download/yellow_tripdata_2024-01.parquet
File exists: True
Size: 47.65 MB


In [46]:
df = pd.read_parquet(FILE_PATH)

In [47]:
print(f"{df.shape[0]} rows, {df.shape[1]} columns")
print(df.columns.tolist())
df.head()

2964624 rows, 19 columns
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [48]:
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

`VendorID` - right type, category
`pickup`, `dropoff` - right type, date recognized
`passenger_count` - wrong type, **possible NaN values**
`trip_distance` - right type
`RatecodeID` - wrong type, **possible NaN values**
`store_and_fwd_flag` - category
`PULocationID` - right type, location coordinates
`DOLocationID` - right type, location, coordinates
`payment_type` - right type, category
`fare_amount`, `extra`, `tip_amount`, `tolls_amount`, `total_amount` - right type, money


In [49]:
categorical_columns = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
]

for col in categorical_columns:
    print("=" * 66 + f"\n")
    print(df[col].value_counts(dropna=False).sort_index())
    print(f"\n")


VendorID
1     729732
2    2234632
6        260
Name: count, dtype: int64



RatecodeID
1.0     2663350
2.0       98713
3.0        7954
4.0        6365
5.0       19410
6.0           7
99.0      28663
NaN      140162
Name: count, dtype: int64



store_and_fwd_flag
N      2813126
Y        11336
NaN     140162
Name: count, dtype: int64



payment_type
0     140162
1    2319046
2     439191
3      19597
4      46628
Name: count, dtype: int64




`VendorID` - contains unexpected value: 6
`payment_type` - is 0 right value?
`payment_type`, `store_and_fwd_flag`, `payment_type` - same amount of NaN values (140162)
This suggests these records may originate from a different ingestion process or have incomplete metadata.

In [50]:
mask = (
    df["RatecodeID"].isna() & df["store_and_fwd_flag"].isna() & (df["payment_type"] == 0)
)
print(mask.sum())

df.loc[mask].head()

140162


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2824462,2,2024-01-01 00:34:19,2024-01-01 00:51:22,NaN,2.04,NaN,NaN,143,141,0,12.72,0.0,0.5,0.00,0.0,1.0,16.72,NaN,NaN
2824463,1,2024-01-01 00:14:31,2024-01-01 00:19:29,NaN,1.60,NaN,NaN,236,238,0,9.30,1.0,0.5,2.86,0.0,1.0,17.16,NaN,NaN
2824464,1,2024-01-01 00:35:11,2024-01-01 01:13:40,NaN,0.00,NaN,NaN,142,79,0,21.01,0.0,0.5,0.00,0.0,1.0,25.01,NaN,NaN
2824465,1,2024-01-01 00:33:37,2024-01-01 00:50:34,NaN,0.00,NaN,NaN,237,4,0,17.79,0.0,0.5,0.00,0.0,1.0,21.79,NaN,NaN
2824466,1,2024-01-01 00:49:04,2024-01-01 01:01:16,NaN,0.00,NaN,NaN,244,50,0,34.65,0.0,0.5,0.00,0.0,1.0,38.65,NaN,NaN


In [51]:
df.isna().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          140162
trip_distance                 0
RatecodeID               140162
store_and_fwd_flag       140162
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     140162
Airport_fee              140162
dtype: int64

## Observations

### VendorID
- Contains three unique values: 1, 2 and 6.
- VendorID = 6 occurs only 260 times and should be verified.

### Missing values pattern
Exactly 140162 records contain missing values in:
- passenger_count
- RatecodeID
- store_and_fwd_flag
- congestion_surcharge
- Airport_fee

The same records also contain:

- payment_type = 0
This indicates a consistent missing-data pattern rather than random missing values.

### Trip data

Some trips have:
- trip_distance = 0
- positive fare_amount
- positive total_amount

These records should be investigated before deciding whether they are invalid.

In [52]:
normal_trips = df.loc[~mask]
special_trips = df.loc[mask]

In [54]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=False)
})

summary

,dtype,missing,missing_%,unique
VendorID,int32,0,0.00,3
tpep_pickup_datetime,datetime64[us],0,0.00,1575706
tpep_dropoff_datetime,datetime64[us],0,0.00,1574780
passenger_count,float64,140162,4.73,11
trip_distance,float64,0,0.00,4489
RatecodeID,float64,140162,4.73,8
store_and_fwd_flag,str,140162,4.73,3
PULocationID,int32,0,0.00,260
DOLocationID,int32,0,0.00,261
payment_type,int64,0,0.00,5


In [ ]:
for col in numeric_columns:
    print("=" * 77)
    print({col})
    print("min:", df[col].min(), "max:", df[col].max())